# Cruzamento Espacial do Voto (VAA)

Neste notebook, aplicamos a Teoria Espacial do Voto de Anthony Downs (1957). Mapeamos eleitores e candidatos como pontos em um espaço multidimensional e calculamos as distâncias matemáticas entre eles.

## Objetivos
1. Definir o "Gabarito" dos pré-candidatos.
2. Calcular Distância Euclidiana e Similaridade por Cosseno.
3. Projetar os dados em 2D usando PCA.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import euclidean_distances, cosine_similarity
from sklearn.decomposition import PCA

sns.set_theme(style='whitegrid')

## 1. Definindo o Perfil dos Pré-Candidatos
Para fins matemáticos, convertemos as posições públicas de candidatos em coordenadas nos nossos blocos D (Economia) e E (Valores).

In [ ]:
candidatos_data = {
    'Candidato': ['Lula', 'Flavio Bolsonaro', 'Romeu Zema', 'Ciro Gomes'],
    'bloco_d_estatais': [1, 5, 5, 2],    # 1=Discorda(Estado forte), 5=Concorda(Privatização)
    'bloco_d_tabelamento': [4, 1, 1, 3], # 1=Livre Mercado, 5=Intervenção Estatal
    'bloco_e_punitivismo': [2, 5, 4, 3], # 1=Garantismo, 5=Punitivismo
    'bloco_e_educacao': [2, 5, 4, 2]     # 1=Progressista, 5=Conservador
}
df_candidatos = pd.DataFrame(candidatos_data).set_index('Candidato')
display(df_candidatos)

## 2. Vetorizando os Eleitores e Calculando Proximidade
Vamos importar nosso dataset e extrair apenas as colunas numéricas de opinião para formar o vetor do eleitor. Faremos o cálculo para os 5 primeiros eleitores como demonstração.

In [ ]:
df_eleitores = pd.read_csv('../data/raw/mock_voters.csv')
cols_opiniao = ['bloco_d_estatais', 'bloco_d_tabelamento', 'bloco_e_punitivismo', 'bloco_e_educacao']

vetores_eleitores = df_eleitores[cols_opiniao].head(5).values
vetores_candidatos = df_candidatos.values

# --- 2.1 Distância Euclidiana ---
# Quanto MENOR, mais próximo
dist_euclidiana = euclidean_distances(vetores_eleitores, vetores_candidatos)

# --- 2.2 Similaridade Cosseno ---
# Quanto MAIOR (perto de 1), mais similar a direção ideológica
sim_cosseno = cosine_similarity(vetores_eleitores, vetores_candidatos)

print('Distância Euclidiana (Eleitor 0 aos Candidatos):')
print(pd.Series(dist_euclidiana[0], index=df_candidatos.index).sort_values())

print('\nSimilaridade Cosseno (Eleitor 0 aos Candidatos):')
print(pd.Series(sim_cosseno[0], index=df_candidatos.index).sort_values(ascending=False))

## 3. Redução de Dimensionalidade (PCA)
Nosso espaço tem 4 dimensões (as 4 perguntas numéricas). Para visualizar num gráfico 2D, aplicamos o Principal Component Analysis (PCA) para comprimir as informações preservando a maior variância possível.

In [ ]:
pca = PCA(n_components=2)

# Concatenar candidatos e todos os eleitores para treinar o mesmo espaço dimensional
vetores_totais = np.vstack([df_candidatos.values, df_eleitores[cols_opiniao].values])
proj_2d = pca.fit_transform(vetores_totais)

# Separar as projeções
proj_cand = proj_2d[:len(df_candidatos)]
proj_eleit = proj_2d[len(df_candidatos):]

plt.figure(figsize=(10, 8))
# Plotar amostra de eleitores (fundo)
plt.scatter(proj_eleit[:, 0], proj_eleit[:, 1], alpha=0.15, color='gray', label='Eleitores')

# Plotar candidatos (destaque)
cores_cand = ['red', 'blue', 'orange', 'purple']
for i, (idx, row) in enumerate(df_candidatos.iterrows()):
    plt.scatter(proj_cand[i, 0], proj_cand[i, 1], color=cores_cand[i], s=200, edgecolors='black', label=idx)

plt.title('Bússola Eleitoral 2026: Projeção 2D (PCA) de Candidatos e Eleitores')
plt.xlabel('Componente Principal 1 (Eixo Econômico predominante)')
plt.ylabel('Componente Principal 2 (Eixo de Valores predominante)')
plt.legend()
plt.show()